# Three Standardized Sentiment Datasets for LZ78 Complexity Experiments

This notebook demonstrates how to load and explore three standardized sentiment analysis datasets:

1. **IMDB Movie Reviews** - Long-form text reviews (positive/negative)
2. **Amazon Polarity** - Product reviews with title and content (positive/negative)
3. **Tweet Sentiment (TweetEval)** - Short social media text (positive/negative)

All datasets are standardized to a common JSON schema with 'text' and 'label' fields, balanced for equal positive/negative examples, and split into train/test sets.

## What this notebook does:
- Loads the standardized datasets from GitHub (with local fallback)
- Explores the data structure and statistics
- Displays example reviews from each dataset
- Visualizes the distribution of text lengths

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print('Dependencies installed successfully!')

In [ ]:
# Imports
import json
import os
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

print('Imports completed!')

In [ ]:
# Data loading helper with GitHub URL and local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-a81b4f-lz78-complexity-for-rule-based-sentiment/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            print(f"Loaded data from GitHub URL")
            return data
    except Exception as e:
        print(f"GitHub URL failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            data = json.load(f)
            print(f"Loaded data from local file")
            return data
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined!')

In [ ]:
# Load the data
data = load_data()

print(f"Number of datasets: {len(data['datasets'])}")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

## Understanding the Data Structure

Each dataset contains examples with the following fields:
- **input**: The text of the review/comment
- **output**: The sentiment label ('positive' or 'negative')
- **metadata_split**: Whether the example is from 'train' or 'test' split
- **metadata_row_index**: The index of the example within its split

Let's explore the structure and display some example reviews.

In [ ]:
# Explore the data structure
for dataset in data['datasets']:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset['dataset']}")
    print(f"{'='*60}")
    
    examples = dataset['examples']
    
    # Count splits
    splits = Counter([ex['metadata_split'] for ex in examples])
    print(f"Splits: {dict(splits)}")
    
    # Count labels
    labels = Counter([ex['output'] for ex in examples])
    print(f"Labels: {dict(labels)}")
    
    # Show first example
    print(f"\nFirst example:")
    print(f"  Input (truncated): {examples[0]['input'][:100]}...")
    print(f"  Output: {examples[0]['output']}")
    print(f"  Split: {examples[0]['metadata_split']}")

## Example Reviews from Each Dataset

Let's display some example reviews from each dataset to understand the differences in text style and length across domains (movies, products, social media).

In [ ]:
# Display example reviews
for dataset in data['datasets']:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset['dataset']}")
    print(f"{'='*60}")
    
    # Get one positive and one negative example
    examples = dataset['examples']
    positive_ex = [ex for ex in examples if ex['output'] == 'positive'][0]
    negative_ex = [ex for ex in examples if ex['output'] == 'negative'][0]
    
    print(f"\nPositive example (first 200 chars):")
    print(f"  {positive_ex['input'][:200]}...")
    
    print(f"\nNegative example (first 200 chars):")
    print(f"  {negative_ex['input'][:200]}...")

## Text Length Analysis

Different datasets have different text length characteristics:
- **IMDB**: Long-form movie reviews (hundreds of words)
- **Amazon Polarity**: Medium-length product reviews
- **Tweet Sentiment**: Short social media posts (limited to ~280 characters)

Let's visualize the distribution of text lengths for each dataset.

In [ ]:
# Analyze text lengths
fig, axes = plt.subplots(1, len(data['datasets']), figsize=(15, 5))

for idx, dataset in enumerate(data['datasets']):
    examples = dataset['examples']
    text_lengths = [len(ex['input'].split()) for ex in examples]  # Word count
    
    ax = axes[idx] if len(data['datasets']) > 1 else axes
    ax.hist(text_lengths, bins=20, edgecolor='black')
    ax.set_title(f"{dataset['dataset']}\n(Word Counts)")
    ax.set_xlabel('Number of Words')
    ax.set_ylabel('Frequency')
    
    print(f"\n{dataset['dataset']}:")
    print(f"  Mean word count: {np.mean(text_lengths):.1f}")
    print(f"  Median word count: {np.median(text_lengths):.1f}")
    print(f"  Min word count: {np.min(text_lengths)}")
    print(f"  Max word count: {np.max(text_lengths)}")

plt.tight_layout()
plt.show()

## Summary Statistics

Let's create a summary table showing the key statistics for each dataset.

In [ ]:
# Create summary statistics table
print(f"{'Dataset':<20} {'Total':<10} {'Train':<10} {'Test':<10} {'Pos':<10} {'Neg':<10}")
print('='*70)

for dataset in data['datasets']:
    examples = dataset['examples']
    
    total = len(examples)
    train = len([ex for ex in examples if ex['metadata_split'] == 'train'])
    test = len([ex for ex in examples if ex['metadata_split'] == 'test'])
    pos = len([ex for ex in examples if ex['output'] == 'positive'])
    neg = len([ex for ex in examples if ex['output'] == 'negative'])
    
    print(f"{dataset['dataset']:<20} {total:<10} {train:<10} {test:<10} {pos:<10} {neg:<10}")

print('\nDataset exploration complete!')